In [1]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 13.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.9 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.

In [ ]:

import os
import json
import random
import sqlite3
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer

# ── Config ────────────────────────────────────────────────────────────────────
DATABASE_PATH = "/kaggle/input/datasets/sutirthajana107/runbugrun/runbugrun.db"  # your database path
MODEL_NAME    = "Qwen/Qwen2.5-Coder-3B-Instruct"
OUTPUT_DIR    = "/kaggle/working/finetuned_model"
DATASET_DIR   = "/kaggle/working/dataset"

LANGUAGES     = [0, 1]       # [] = all languages

# ── BIT PRECISION ──────────────────────────────────────────────────────────────
# Change this single value to compare results: 4 or 8 only.
# IMPORTANT: bitsandbytes (the library used below) does NOT support true 2-bit
# quantization. Only 4-bit and 8-bit are valid here. Real 2-bit quantization
# needs a completely different library (e.g. AutoGPTQ / AWQ / HQQ) and a
# different model-loading setup — it is not a simple value swap like this one.
BIT_PRECISION = 4

# ── Train / Validation / Test split ───────────────────────────────────────────
# train_data → model learns from this (in chunks, see below)
# val_data   → checked DURING training for early stopping / best checkpoint
# test_data  → held out completely, never touched until final evaluation
TRAIN_SPLIT = 0.8
VAL_SPLIT   = 0.1
TEST_SPLIT  = 0.1
SPLIT_SEED  = 42   # fixed seed → same split every run → test set never leaks into training

# ── Chunked / batch-wise training ─────────────────────────────────────────────
# Your full filtered dataset is large. Training all of it in one Kaggle
# session is too slow, but training on a tiny MAX_SAMPLES (like 100) gives
# low accuracy. The fix: train in CHUNK_SIZE-sized batches of the train pool,
# one chunk per session, and resume from the previous chunk's saved weights
# by increasing RESUME_FROM_CHUNK next time. This way you eventually train on
# the full large dataset, just spread across multiple sessions instead of
# one slow run.
CHUNK_SIZE        = 2000    # samples to train THIS session (raise/lower as needed)
VAL_SAMPLE_SIZE   = 100     # how many validation samples to actually use per eval pass.
                            # val_full itself is 10% of the WHOLE database (could be
                            # thousands of rows) — evaluating on all of it every
                            # eval_steps is what made one eval pass take 10+ hours.
                            # This caps it so eval stays fast regardless of CHUNK_SIZE.
RESUME_FROM_CHUNK = 0       # 0 = first session, 1 = second session, etc.

MAX_SEQ_LEN   = 1024   # IMPORTANT: measured directly from real data — median
                       # sample length was ~960 tokens, with some extreme
                       # outliers (one was 81,073 tokens!). At 256, literally
                       # 0% of samples fit and every example was being
                       # truncated before reaching the "Fixed Code" section —
                       # the model was learning from broken/incomplete data.
                       # 1024 covers the majority of realistic samples.
                       # Extreme outliers are filtered out separately below
                       # (MAX_FILTER_LEN) so they don't waste compute or get
                       # silently truncated either.
MAX_FILTER_LEN = 1500  # any sample longer than this (in tokens) is DROPPED
                       # entirely from train/val/test, rather than truncated.
                       # This removes rare extreme outliers (like the 81k-token
                       # sample) that would otherwise force huge padding/compute
                       # cost on every batch they're part of.
NUM_EPOCHS    = 50   # high ceiling — early stopping (Cell 6) will stop training
                      # automatically once validation loss stops improving, so
                      # this rarely runs to completion. It's a safety cap only.
EARLY_STOPPING_PATIENCE = 3   # stop after 3 eval checks with no improvement
BATCH_SIZE    = 4    # raised from 2 — with shorter MAX_SEQ_LEN there's
                      # more VRAM headroom, and a bigger batch means fewer
                      # total steps for the same effective batch size
GRAD_ACCUM    = 2    # lowered from 4 — keeps effective batch size the same
                      # (4*2=8, same as before: 2*4=8) but with fewer micro-steps
LEARNING_RATE = 2e-4
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
LORA_TARGETS  = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

LANGUAGE_MAP  = {
    0: "C", 1: "C++", 2: "JavaScript", 3: "Java",
    4: "Ruby", 5: "Python", 6: "PHP", 7: "Go", 8: "C#"
}

print("✅ Config done!")
print(f"   CUDA available : {torch.cuda.is_available()}")
print(f"   GPU            : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"   Bit precision  : {BIT_PRECISION}-bit")
print(f"   Chunk          : {RESUME_FROM_CHUNK}  ({CHUNK_SIZE} samples/session)")


In [ ]:
import json
import os
from datetime import datetime

LOG_FILE = "/kaggle/working/experiment_log.json"  # ← put in Google Drive / permanent path if possible

def load_log():
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, "r") as f:
            return json.load(f)
    return {"experiments": []}

def save_log(log_data):
    with open(LOG_FILE, "w") as f:
        json.dump(log_data, f, indent=2, ensure_ascii=False)

def log_event(stage, details: dict, notes: str = ""):
    """
    Add one entry to the logbook.

    stage   : short name e.g. "SFT_TRAINING", "EVAL_BEFORE_AFTER", "DPO_TRAINING"
    details : any dict of useful numbers/settings to remember
    notes   : free text note to yourself
    """
    log_data = load_log()
    entry = {
        "timestamp": datetime.now().isoformat(),
        "stage"    : stage,
        "details"  : details,
        "notes"    : notes
    }
    log_data["experiments"].append(entry)
    save_log(log_data)
    print(f"📝 Logged: [{stage}] at {entry['timestamp']}")

def show_log(stage_filter=None):
    """Print the full logbook, optionally filtered by stage."""
    log_data = load_log()
    experiments = log_data["experiments"]
    if stage_filter:
        experiments = [e for e in experiments if e["stage"] == stage_filter]

    print("\n" + "=" * 70)
    print(f"  📖 EXPERIMENT LOGBOOK  ({len(experiments)} entries)")
    print("=" * 70)
    for i, e in enumerate(experiments, 1):
        print(f"\n[{i}] {e['stage']}  —  {e['timestamp']}")
        for k, v in e["details"].items():
            print(f"     {k}: {v}")
        if e["notes"]:
            print(f"     📌 Note: {e['notes']}")
    print("=" * 70)

print("✅ Logging system ready! Log file:", LOG_FILE)
show_log()  # shows existing logs if any (empty first time)

In [ ]:
log_event("SESSION_START", {
        "model": MODEL_NAME, "languages": LANGUAGES, "bit_precision": BIT_PRECISION,
        "chunk": RESUME_FROM_CHUNK, "chunk_size": CHUNK_SIZE,
        "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE,
        "lora_r": LORA_R
    }, notes="Starting SFT run")

show_log()  # shows everything logged so far, across all previous sessions too


In [ ]:
# Tokenizer is needed here (Cell 4) to measure/filter sample lengths, before
# Cell 5 loads the full model. This is lightweight — just the tokenizer,
# not the model — so loading it here doesn't duplicate heavy GPU work.
if "tokenizer" not in dir():
    print("🔤 Loading tokenizer (needed for length filtering)...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = "right"

def build_language_filter():
    if not LANGUAGES:
        return "1=1", []
    elif len(LANGUAGES) == 1:
        return "language = ?", list(LANGUAGES)
    else:
        placeholders = ", ".join("?" * len(LANGUAGES))
        return f"language IN ({placeholders})", list(LANGUAGES)

def format_prompt(buggy_code, fixed_code, problem_statement, language_code):
    lang = LANGUAGE_MAP.get(language_code, "code")
    return f"""### Instruction:
You are an expert {lang} developer. Fix the bug in the following {lang} code.

### Problem:
{problem_statement}

### Buggy Code:
{buggy_code}

### Fixed Code:
{fixed_code}"""

def fetch_and_prepare():
    """
    Builds a train/val/test split using a FIXED seed (SPLIT_SEED), so the
    test set is always the exact same rows every time this runs — meaning it
    never overlaps with whatever gets trained on, even across sessions.

    Then selects only THIS session's CHUNK from the train pool, so you can
    train on a large dataset gradually across multiple sessions instead of
    needing to fit it all into one slow run.
    """
    # Reuse already-built split if it exists this session (avoids re-querying DB
    # and, importantly, avoids re-shuffling — which would change the test set)
    if os.path.exists(f"{DATASET_DIR}/test_raw.json") and os.path.exists(f"{DATASET_DIR}/train_full.json"):
        print("✅ Split already exists this session — reusing saved files.")
        with open(f"{DATASET_DIR}/train_full.json") as f:
            train_full = json.load(f)
        with open(f"{DATASET_DIR}/val.json") as f:
            val_full = json.load(f)
    else:
        print("📦 Fetching data from database...")
        conn   = sqlite3.connect(DATABASE_PATH)
        cursor = conn.cursor()

        lang_filter, lang_params = build_language_filter()
        query = f"""
            WITH filtered_bugs AS (
                SELECT id, buggy_code, fixed_code, language, problem_id
                FROM bugs
                WHERE {lang_filter}
                ORDER BY id ASC
            )
            SELECT filtered_bugs.*, problems.text AS problem_statement
            FROM filtered_bugs
            JOIN problems ON problems.problem_id = filtered_bugs.problem_id
        """
        cursor.execute(query, lang_params)
        columns = [desc[0] for desc in cursor.description]
        rows    = [dict(zip(columns, row)) for row in cursor.fetchall()]
        conn.close()
        print(f"   ✅ Total records: {len(rows)}")

        # Fixed-seed shuffle → same train/val/test split every time this is run,
        # so the held-out test set never accidentally ends up in training data.
        rng = random.Random(SPLIT_SEED)
        shuffled = rows[:]
        rng.shuffle(shuffled)

        n         = len(shuffled)
        train_end = int(n * TRAIN_SPLIT)
        val_end   = train_end + int(n * VAL_SPLIT)

        train_rows = shuffled[:train_end]
        val_rows   = shuffled[train_end:val_end]
        test_rows  = shuffled[val_end:]

        print("\n✍️  Formatting dataset...")
        train_full_unfiltered = [{"text": format_prompt(r["buggy_code"], r["fixed_code"],
                        r["problem_statement"], r["language"]), "id": r["id"]} for r in train_rows]
        val_full_unfiltered   = [{"text": format_prompt(r["buggy_code"], r["fixed_code"],
                        r["problem_statement"], r["language"]), "id": r["id"]} for r in val_rows]

        # ── Filter out extreme outliers (token length check) ─────────────────
        # Measured on real data: median sample is ~960 tokens, but a few
        # outliers reached 80,000+ tokens. Those outliers would either get
        # silently truncated (losing the fix entirely) or force huge padding
        # cost on every batch they're in. Dropping them entirely is safer and
        # faster than truncating.
        print(f"🔍 Filtering samples longer than {MAX_FILTER_LEN} tokens...")

        def batch_filter_by_length(items, get_text_fn, batch_size=1000):
            """
            Tokenizes in BATCHES instead of one item at a time. With 479,129
            total records, calling tokenizer() once per item in a Python loop
            is extremely slow (Python call overhead × hundreds of thousands).
            Batch tokenization lets the underlying fast Rust tokenizer process
            many texts in one call, which is dramatically faster.
            """
            keep = []
            total = len(items)
            for i in range(0, total, batch_size):
                batch = items[i:i + batch_size]
                texts = [get_text_fn(x) for x in batch]
                encoded = tokenizer(texts, truncation=False)["input_ids"]
                for item, tokens in zip(batch, encoded):
                    if len(tokens) <= MAX_FILTER_LEN:
                        keep.append(item)
                if (i // batch_size) % 20 == 0:
                    print(f"      ...processed {min(i + batch_size, total)}/{total}")
            return keep

        train_full = batch_filter_by_length(train_full_unfiltered, lambda x: x["text"])
        val_full   = batch_filter_by_length(val_full_unfiltered, lambda x: x["text"])

        dropped_train = len(train_full_unfiltered) - len(train_full)
        dropped_val   = len(val_full_unfiltered) - len(val_full)
        print(f"   Dropped {dropped_train} long samples from train ({dropped_train/len(train_full_unfiltered)*100:.1f}%)")
        print(f"   Dropped {dropped_val} long samples from val ({dropped_val/max(len(val_full_unfiltered),1)*100:.1f}%)")

        # Also filter test_rows the same way (format on the fly just to check length,
        # but keep the original raw row — test_raw.json stores unformatted rows)
        test_rows = batch_filter_by_length(
            test_rows,
            lambda r: format_prompt(r["buggy_code"], r["fixed_code"], r["problem_statement"], r["language"])
        )
        dropped_test_count = len(shuffled[val_end:]) - len(test_rows)
        print(f"   Dropped {dropped_test_count} long samples from test")

        os.makedirs(DATASET_DIR, exist_ok=True)
        with open(f"{DATASET_DIR}/train_full.json", "w") as f:
            json.dump(train_full, f)
        with open(f"{DATASET_DIR}/val.json", "w") as f:
            json.dump(val_full, f)
        with open(f"{DATASET_DIR}/test_raw.json", "w") as f:
            json.dump(test_rows, f)   # raw rows (not text-formatted), used later for evaluation

        print(f"   Train pool : {len(train_full)}  (this gets split into chunks below)")
        print(f"   Val        : {len(val_full)}  (used DURING training for early stopping)")
        print(f"   Test       : {len(test_rows)}  (held out — NEVER used in training)")

    # ── Select this session's chunk from the train pool ──────────────────────
    chunk_start = RESUME_FROM_CHUNK * CHUNK_SIZE
    chunk_end   = chunk_start + CHUNK_SIZE
    train_data  = train_full[chunk_start:chunk_end]

    # Cap validation set size so eval doesn't take hours regardless of CHUNK_SIZE.
    # val_full is 10% of the ENTIRE database (could be thousands of rows) — using
    # all of it every eval_steps was why one evaluation pass took 10+ hours.
    # VAL_SAMPLE_SIZE caps how many of those rows are actually used for eval.
    eval_data = val_full[:VAL_SAMPLE_SIZE]

    total_chunks = (len(train_full) // CHUNK_SIZE) + 1
    print(f"\n📦 This session's chunk: {RESUME_FROM_CHUNK}  (rows {chunk_start}-{chunk_end} of {len(train_full)})")
    print(f"   Train samples this session : {len(train_data)}")
    print(f"   Val samples (fixed)        : {len(eval_data)}")
    print(f"   Sessions needed to cover everything: ~{total_chunks}")
    if len(train_data) == 0:
        print("   ⚠️  This chunk is EMPTY — you've already covered the full train pool!")
    print(f"   👉 Next session: set RESUME_FROM_CHUNK = {RESUME_FROM_CHUNK + 1} in Cell 1")

    return train_data, eval_data

train_data, eval_data = fetch_and_prepare()

log_event("DATASET_CHUNK", {
        "chunk": RESUME_FROM_CHUNK, "train_samples": len(train_data),
        "eval_samples": len(eval_data), "languages": LANGUAGES
    })


In [ ]:
if "tokenizer" not in dir():
    print("🔤 Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.padding_side = "right"
else:
    print("🔤 Tokenizer already loaded (from Cell 4) — reusing it.")

print(f"🤖 Loading model in {BIT_PRECISION}-bit (QLoRA)...")

if BIT_PRECISION == 4:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_quant_type       = "nf4",
        bnb_4bit_compute_dtype    = torch.bfloat16,
        bnb_4bit_use_double_quant = True,
    )
elif BIT_PRECISION == 8:
    bnb_config = BitsAndBytesConfig(
        load_in_8bit = True,
    )
else:
    raise ValueError(
        f"BIT_PRECISION={BIT_PRECISION} is not supported by bitsandbytes. "
        f"Only 4 or 8 are valid here. True 2-bit quantization needs a "
        f"different library (AutoGPTQ / AWQ / HQQ) and a different "
        f"model-loading setup — it isn't a simple value change like this."
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True
)
model = prepare_model_for_kbit_training(model)

# Resume from previous chunk's saved LoRA weights if this isn't the first session
PREV_CHUNK_DIR = f"{OUTPUT_DIR}_chunk{RESUME_FROM_CHUNK - 1}_{BIT_PRECISION}bit"
if RESUME_FROM_CHUNK > 0 and os.path.exists(PREV_CHUNK_DIR):
    print(f"🔁 Resuming LoRA weights from previous chunk: {PREV_CHUNK_DIR}")
    model = PeftModel.from_pretrained(model, PREV_CHUNK_DIR, is_trainable=True)
else:
    print("🔧 Applying fresh LoRA adapters...")
    lora_config = LoraConfig(
        r              = LORA_R,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        target_modules = LORA_TARGETS,
        bias           = "none",
        task_type      = "CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


In [ ]:
from trl import SFTConfig

train_dataset = Dataset.from_list(train_data)
eval_dataset  = Dataset.from_list(eval_data)

training_args = SFTConfig(
    output_dir                  = f"{OUTPUT_DIR}_chunk{RESUME_FROM_CHUNK}_{BIT_PRECISION}bit_checkpoints",
    max_length                  = MAX_SEQ_LEN,   # ← actually enforces sequence length now
                                                   #   (previous code defined MAX_SEQ_LEN but
                                                   #   never passed it anywhere, so the trainer
                                                   #   fell back to the tokenizer's own much
                                                   #   longer default — a real cause of slowness)
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_steps                = 100,
    lr_scheduler_type           = "cosine",
    fp16                        = False,
    bf16                        = True,
    logging_steps               = 10,
    save_steps                  = 100,
    eval_steps                  = 100,
    eval_strategy               = "steps",
    save_total_limit            = 2,
    load_best_model_at_end      = True,   # picks best checkpoint based on val loss
    metric_for_best_model       = "eval_loss",   # required by EarlyStoppingCallback
    greater_is_better            = False,         # lower eval_loss = better
    report_to                   = "none",
    optim                       = "paged_adamw_8bit",
)

trainer = SFTTrainer(
    model          = model,
    train_dataset  = train_dataset,
    eval_dataset   = eval_dataset,
    args           = training_args,
    callbacks      = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    # ↑ Stops training automatically once val_loss hasn't improved for
    #   EARLY_STOPPING_PATIENCE eval checks in a row, instead of always
    #   running the full NUM_EPOCHS. NUM_EPOCHS is just a high safety ceiling.
)

print(f"🏋️  Starting training (early stopping after {EARLY_STOPPING_PATIENCE} checks with no improvement)...")
trainer.train()

# ── Per-epoch results ──────────────────────────────────────────────────────────
epoch_results = {}
for entry in trainer.state.log_history:
    if "epoch" in entry:
        ep = round(entry["epoch"], 2)
        epoch_results.setdefault(ep, {})
        if "loss" in entry:
            epoch_results[ep]["train_loss"] = round(entry["loss"], 4)
        if "eval_loss" in entry:
            epoch_results[ep]["val_loss"] = round(entry["eval_loss"], 4)

print("\n📊 Per-epoch results:")
for ep, metrics in sorted(epoch_results.items()):
    print(f"   Epoch {ep}: {metrics}")


In [ ]:
CURRENT_SAVE_DIR = f"{OUTPUT_DIR}_chunk{RESUME_FROM_CHUNK}_{BIT_PRECISION}bit"

print("💾 Saving model...")
trainer.save_model(CURRENT_SAVE_DIR)
tokenizer.save_pretrained(CURRENT_SAVE_DIR)
print(f"✅ Model saved to {CURRENT_SAVE_DIR}")
print(f"👉 Next session: set RESUME_FROM_CHUNK = {RESUME_FROM_CHUNK + 1} in Cell 1")
print("🎉 Fine-tuning complete!")

log_event("SFT_TRAINING_DONE", {
        "output_dir": CURRENT_SAVE_DIR, "chunk": RESUME_FROM_CHUNK,
        "bit_precision": BIT_PRECISION, "per_epoch_results": epoch_results
    }, notes="SFT model saved")


In [8]:
!pip install -q openai

In [9]:
import json
import time
import re
import sqlite3
import torch
from openai import OpenAI
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ── Settings ──────────────────────────────────────────────────────────────────
OPENROUTER_API_KEY = "csk-ehf83xj5r49p2xnw32ejvv4yxkcphvktrmx92pj48ecjtvhw"   # ← cerebras key
EVAL_MODEL         = "gpt-oss-120b"                  # ← cerebras model
EVAL_DELAY         = 35                                # ← seconds between calls
TEST_SAMPLES       = 20                                # ← how many bugs to test

BEFORE_RESULTS_FILE    = "/kaggle/working/before_results.json"
AFTER_RESULTS_FILE     = "/kaggle/working/after_results.json"
BEFORE_EVAL_FILE       = "/kaggle/working/before_evaluation.json"
AFTER_EVAL_FILE        = "/kaggle/working/after_evaluation.json"
COMPARISON_FILE        = "/kaggle/working/comparison.json"

print("✅ Evaluation config done!")

✅ Evaluation config done!


In [ ]:
def fetch_test_samples(n=TEST_SAMPLES):
    """
    Loads from the held-out test_raw.json built in Cell 4, NOT a fresh DB query.
    This guarantees these rows were never seen during training or validation —
    the previous version queried "first N rows by id", which very likely
    overlapped with the training data and made the before/after comparison
    unreliable.
    """
    with open(f"{DATASET_DIR}/test_raw.json") as f:
        test_rows = json.load(f)
    rng = random.Random(123)   # different seed, just for picking order within test set
    shuffled = test_rows[:]
    rng.shuffle(shuffled)
    selected = shuffled[:n]
    print(f"✅ Loaded {len(selected)} samples from the HELD-OUT test set")
    print(f"   ({len(test_rows)} total test rows, none were used in training or validation)")
    return selected

test_samples = fetch_test_samples()


In [ ]:
def generate_fix(model, tokenizer, buggy_code, problem_statement, language_code):
    lang = LANGUAGE_MAP.get(language_code, "code")
    prompt = f"""### Instruction:
You are an expert {lang} developer. Fix the bug in the following {lang} code.

### Problem:
{problem_statement}

### Buggy Code:
{buggy_code}

### Fixed Code:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 256,
            temperature    = 0.1,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Load original model
print("🤖 Loading ORIGINAL model (before training)...")
orig_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
orig_model     = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype       = torch.bfloat16,
    device_map        = "auto",
    trust_remote_code = True
)
orig_model.eval()
print("✅ Original model loaded!")

# Generate fixes with original model
print("\n🔧 Generating fixes with ORIGINAL model...")
before_results = []
for i, sample in enumerate(test_samples, 1):
    print(f"   [{i}/{len(test_samples)}] Bug ID: {sample['id']} ...", end=" ", flush=True)
    try:
        fixed = generate_fix(
            orig_model, orig_tokenizer,
            sample["buggy_code"],
            sample["problem_statement"],
            sample["language"]
        )
        print("✅")
        before_results.append({
            "bug_id"           : sample["id"],
            "language_code"    : sample["language"],
            "language_name"    : LANGUAGE_MAP.get(sample["language"], "code"),
            "problem_statement": sample["problem_statement"],
            "buggy_code"       : sample["buggy_code"],
            "fixed_code_ref"   : sample["fixed_code"],
            "fixed_code_llm"   : fixed,
            "model"            : "original"
        })
    except Exception as e:
        print(f"❌ {e}")

with open(BEFORE_RESULTS_FILE, "w") as f:
    json.dump(before_results, f, indent=2)
print(f"\n✅ Before results saved to {BEFORE_RESULTS_FILE}")

# Free memory
del orig_model
torch.cuda.empty_cache()
print("🧹 Original model removed from GPU memory")

In [ ]:
print("🤖 Loading FINE-TUNED model (after training)...")
ft_tokenizer = AutoTokenizer.from_pretrained(CURRENT_SAVE_DIR, trust_remote_code=True)
ft_base      = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype       = torch.bfloat16,
    device_map        = "auto",
    trust_remote_code = True
)
ft_model = PeftModel.from_pretrained(ft_base, CURRENT_SAVE_DIR)
ft_model.eval()
print("✅ Fine-tuned model loaded!")

# Generate fixes with fine-tuned model
print("\n🔧 Generating fixes with FINE-TUNED model...")
after_results = []
for i, sample in enumerate(test_samples, 1):
    print(f"   [{i}/{len(test_samples)}] Bug ID: {sample['id']} ...", end=" ", flush=True)
    try:
        fixed = generate_fix(
            ft_model, ft_tokenizer,
            sample["buggy_code"],
            sample["problem_statement"],
            sample["language"]
        )
        print("✅")
        after_results.append({
            "bug_id"           : sample["id"],
            "language_code"    : sample["language"],
            "language_name"    : LANGUAGE_MAP.get(sample["language"], "code"),
            "problem_statement": sample["problem_statement"],
            "buggy_code"       : sample["buggy_code"],
            "fixed_code_ref"   : sample["fixed_code"],
            "fixed_code_llm"   : fixed,
            "model"            : "finetuned"
        })
    except Exception as e:
        print(f"❌ {e}")

with open(AFTER_RESULTS_FILE, "w") as f:
    json.dump(after_results, f, indent=2)
print(f"\n✅ After results saved to {AFTER_RESULTS_FILE}")

# Free memory
del ft_model
torch.cuda.empty_cache()
print("🧹 Fine-tuned model removed from GPU memory")


In [ ]:
def get_gpt_score(client, record):
    prompt = f"""You are an expert code reviewer. Evaluate the quality of a bug fix.

Language: {record.get('language_name', 'code')}

Problem Statement:
{record.get('problem_statement', '')}

Buggy Code:
{record.get('buggy_code', '')}

Reference Fix (correct fix):
{record.get('fixed_code_ref', '')}

LLM Generated Fix (fix to evaluate):
{record.get('fixed_code_llm', '')}

Score the LLM Generated Fix from 0 to 10 based on:
  - Correctness (does it fix the bug?)
  - Logic (same logic as reference?)
  - Code Quality (clean and readable?)

Reply in EXACT format:
SCORE: <number 0-10>
REASON: <one line explanation>
"""
    response = client.chat.completions.create(
        model    = EVAL_MODEL,
        messages = [
            {"role": "system", "content": "You are an expert code reviewer."},
            {"role": "user",   "content": prompt}
        ]
    )
    reply        = response.choices[0].message.content.strip()
    score_match  = re.search(r"SCORE:\s*([\d.]+)", reply)
    reason_match = re.search(r"REASON:\s*(.+)", reply)
    score  = float(score_match.group(1)) if score_match else None
    reason = reason_match.group(1).strip() if reason_match else ""
    if score: score = min(10.0, max(0.0, score))
    return score, reason


client = OpenAI(
    base_url = "https://api.cerebras.ai/v1",
    api_key  = OPENROUTER_API_KEY
)
def get_gpt_score_with_retry(client, record, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            score, reason = get_gpt_score(client, record)
            return score, reason
        except Exception as e:
            err = str(e)
            if "429" in err:
                wait = 35
                match = re.search(r"retry_after_seconds['\"]:\s*([\d.]+)", err)
                if match:
                    wait = int(float(match.group(1))) + 2
                if attempt < max_retries:
                    print(f"\n   ⚠️  Rate limited. Waiting {wait}s... (attempt {attempt}/{max_retries})")
                    time.sleep(wait)
                    print(f"   🔄 Retrying...", end=" ", flush=True)
                else:
                    raise
            else:
                raise
# Score BEFORE results
print("📊 Scoring BEFORE results...")
before_evals = []
for i, record in enumerate(before_results, 1):
    print(f"   [{i}/{len(before_results)}] Bug ID: {record['bug_id']} ...", end=" ", flush=True)
    try:
        score, reason = get_gpt_score(client, record)
        print(f"✅ Score: {score}/10")
        before_evals.append({**record, "gpt_score": score, "reason": reason})
    except Exception as e:
        print(f"❌ {e}")
        before_evals.append({**record, "gpt_score": None, "error": str(e)})
    if i < len(before_results):
        time.sleep(EVAL_DELAY)

with open(BEFORE_EVAL_FILE, "w") as f:
    json.dump(before_evals, f, indent=2)

# Score AFTER results
print("\n📊 Scoring AFTER results...")
after_evals = []
for i, record in enumerate(after_results, 1):
    print(f"   [{i}/{len(after_results)}] Bug ID: {record['bug_id']} ...", end=" ", flush=True)
    try:
        score, reason = get_gpt_score(client, record)
        print(f"✅ Score: {score}/10")
        after_evals.append({**record, "gpt_score": score, "reason": reason})
    except Exception as e:
        print(f"❌ {e}")
        after_evals.append({**record, "gpt_score": None, "error": str(e)})
    if i < len(after_results):
        time.sleep(EVAL_DELAY)

with open(AFTER_EVAL_FILE, "w") as f:
    json.dump(after_evals, f, indent=2)

📊 Scoring BEFORE results...
   [1/20] Bug ID: 183 ... ✅ Score: 2.0/10
   [2/20] Bug ID: 187 ... ✅ Score: 10.0/10
   [3/20] Bug ID: 188 ... ✅ Score: 4.0/10
   [4/20] Bug ID: 201 ... ✅ Score: 4.0/10
   [5/20] Bug ID: 211 ... ✅ Score: 4.0/10
   [6/20] Bug ID: 214 ... ✅ Score: 4.0/10
   [7/20] Bug ID: 218 ... ✅ Score: 2.0/10
   [8/20] Bug ID: 220 ... ✅ Score: 2.0/10
   [9/20] Bug ID: 221 ... ✅ Score: 2.0/10
   [10/20] Bug ID: 224 ... ✅ Score: 4.0/10
   [11/20] Bug ID: 225 ... ✅ Score: 2.0/10
   [12/20] Bug ID: 241 ... ✅ Score: 4.0/10
   [13/20] Bug ID: 242 ... ✅ Score: 2.0/10
   [14/20] Bug ID: 243 ... ✅ Score: 3.0/10
   [15/20] Bug ID: 251 ... ✅ Score: 2.0/10
   [16/20] Bug ID: 257 ... ✅ Score: 1.0/10
   [17/20] Bug ID: 265 ... ✅ Score: 4.0/10
   [18/20] Bug ID: 272 ... ✅ Score: 2.0/10
   [19/20] Bug ID: 284 ... ✅ Score: 3.0/10
   [20/20] Bug ID: 304 ... ✅ Score: 10.0/10

📊 Scoring AFTER results...
   [1/20] Bug ID: 183 ... ✅ Score: 5.0/10
   [2/20] Bug ID: 187 ... ✅ Score: 10.0/10
   [3/

In [ ]:
before_scores = [e["gpt_score"] for e in before_evals if e.get("gpt_score") is not None]
after_scores  = [e["gpt_score"] for e in after_evals  if e.get("gpt_score") is not None]

before_avg = sum(before_scores) / len(before_scores) if before_scores else 0
after_avg  = sum(after_scores)  / len(after_scores)  if after_scores  else 0
improvement = after_avg - before_avg

print("\n" + "=" * 65)
print("  📊 COMPARISON: Before vs After Fine-tuning")
print("=" * 65)
print(f"  Samples tested     : {len(before_scores)}")
print(f"  Before (original)  : {before_avg:.2f} / 10")
print(f"  After  (finetuned) : {after_avg:.2f}  / 10")
print(f"  Improvement        : {improvement:+.2f} points")
print("=" * 65)

if improvement > 0:
    print(f"  ✅ Fine-tuning IMPROVED the model by {improvement:.2f} points!")
elif improvement == 0:
    print(f"  ➡️  No change after fine-tuning")
else:
    print(f"  ⚠️  Fine-tuning decreased performance by {abs(improvement):.2f} points")

# Save comparison
comparison = {
    "samples_tested"  : len(before_scores),
    "before_avg_score": round(before_avg, 2),
    "after_avg_score" : round(after_avg, 2),
    "improvement"     : round(improvement, 2),
}
with open(COMPARISON_FILE, "w") as f:
    json.dump(comparison, f, indent=2)
print(f"\n  💾 Comparison saved to {COMPARISON_FILE}")
print("🎉 Evaluation complete!")

log_event("EVAL_COMPARISON", {
        "samples_tested": len(before_scores),
        "before_avg": round(before_avg, 2),
        "after_avg": round(after_avg, 2),
        "improvement": round(improvement, 2),
        "model_dir": CURRENT_SAVE_DIR, "bit_precision": BIT_PRECISION, "chunk": RESUME_FROM_CHUNK
   }, notes="SFT before vs after GPT score")

In [ ]:
!pip install -q trl peft bitsandbytes

In [ ]:
import json
import sqlite3
import random
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig

# ── DPO Settings ──────────────────────────────────────────────────────────────
SFT_MODEL_DIR   = CURRENT_SAVE_DIR                            # your fine-tuned SFT model (this chunk + bit precision)
DPO_OUTPUT_DIR  = "/kaggle/working/dpo_finetuned_model"        # where DPO model will be saved
DPO_DATASET_DIR = "/kaggle/working/dpo_dataset"

DPO_MAX_SAMPLES = 100     # number of preference pairs (keep small, DPO is slower per sample)
DPO_EPOCHS      = 1
DPO_BATCH_SIZE  = 10
DPO_GRAD_ACCUM  = 4
DPO_LR          = 5e-6     # DPO needs MUCH smaller learning rate than SFT
DPO_BETA        = 0.1      # controls how strongly model follows preference (0.1-0.5 typical)

print("✅ DPO config done!")

In [ ]:
def build_dpo_prompt(buggy_code, problem_statement, language_code):
    lang = LANGUAGE_MAP.get(language_code, "code")
    return f"""### Instruction:
You are an expert {lang} developer. Fix the bug in the following {lang} code.

### Problem:
{problem_statement}

### Buggy Code:
{buggy_code}

### Fixed Code:
"""

def generate_two_fixes(model, tokenizer, prompt):
    """Generate 2 different fixes using different sampling temperatures."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Fix 1: low temperature (more confident/deterministic)
    with torch.no_grad():
        out1 = model.generate(
            **inputs, max_new_tokens=256, temperature=0.2,
            do_sample=True, pad_token_id=tokenizer.eos_token_id
        )
    fix1 = tokenizer.decode(out1[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # Fix 2: high temperature (more random/exploratory)
    with torch.no_grad():
        out2 = model.generate(
            **inputs, max_new_tokens=256, temperature=0.9,
            do_sample=True, pad_token_id=tokenizer.eos_token_id
        )
    fix2 = tokenizer.decode(out2[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    return fix1, fix2


# Fetch samples for building preference dataset
def fetch_dpo_samples(n=DPO_MAX_SAMPLES):
    conn   = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()
    lang_filter, lang_params = build_language_filter()
    query = f"""
        WITH filtered_bugs AS (
            SELECT id, buggy_code, fixed_code, language, problem_id
            FROM bugs
            WHERE {lang_filter}
            ORDER BY RANDOM()
            LIMIT {n}
        )
        SELECT filtered_bugs.*, problems.text AS problem_statement
        FROM filtered_bugs
        JOIN problems ON problems.problem_id = filtered_bugs.problem_id
    """
    cursor.execute(query, lang_params)
    columns = [desc[0] for desc in cursor.description]
    rows    = [dict(zip(columns, row)) for row in cursor.fetchall()]
    conn.close()
    print(f"✅ Fetched {len(rows)} samples for DPO dataset")
    return rows


print("🤖 Loading SFT model to generate preference pairs...")
dpo_tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_DIR, trust_remote_code=True)
dpo_base      = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
)
sft_model_for_gen = PeftModel.from_pretrained(dpo_base, SFT_MODEL_DIR)
sft_model_for_gen.eval()
print("✅ SFT model loaded!")

dpo_raw_samples = fetch_dpo_samples()

In [ ]:
client = OpenAI(
    base_url = "https://api.cerebras.ai/v1",   # or openrouter, whichever you use
    api_key  = OPENROUTER_API_KEY
)

dpo_pairs = []
print("🔧 Generating and scoring preference pairs...")

for i, sample in enumerate(dpo_raw_samples, 1):
    print(f"   [{i}/{len(dpo_raw_samples)}] Bug ID: {sample['id']} ...", end=" ", flush=True)
    try:
        prompt = build_dpo_prompt(sample["buggy_code"], sample["problem_statement"], sample["language"])
        fix1, fix2 = generate_two_fixes(sft_model_for_gen, dpo_tokenizer, prompt)

        record_for_scoring = {
            "language_name"    : LANGUAGE_MAP.get(sample["language"], "code"),
            "problem_statement": sample["problem_statement"],
            "buggy_code"       : sample["buggy_code"],
            "fixed_code_ref"   : sample["fixed_code"],
        }

        score1, _ = get_gpt_score_with_retry(client, {**record_for_scoring, "fixed_code_llm": fix1})
        score2, _ = get_gpt_score_with_retry(client, {**record_for_scoring, "fixed_code_llm": fix2})

        if score1 is None or score2 is None or score1 == score2:
            print("⏭️  skipped (equal/invalid scores)")
            continue

        # Higher score = chosen, lower score = rejected
        if score1 > score2:
            chosen, rejected = fix1, fix2
        else:
            chosen, rejected = fix2, fix1

        dpo_pairs.append({
            "prompt"  : prompt,
            "chosen"  : chosen,
            "rejected": rejected
        })
        print(f"✅ chosen_score={max(score1,score2)} rejected_score={min(score1,score2)}")

        time.sleep(EVAL_DELAY)  # avoid rate limit

    except Exception as e:
        print(f"❌ {e}")

print(f"\n✅ Built {len(dpo_pairs)} preference pairs")

# Save preference dataset
os.makedirs(DPO_DATASET_DIR, exist_ok=True)
with open(f"{DPO_DATASET_DIR}/dpo_pairs.json", "w") as f:
    json.dump(dpo_pairs, f, indent=2)
print(f"💾 Saved to {DPO_DATASET_DIR}/dpo_pairs.json")

# Free SFT generation model from memory
del sft_model_for_gen, dpo_base
torch.cuda.empty_cache()
print("🧹 Cleared GPU memory")

In [ ]:
print("🤖 Loading model for DPO training...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

dpo_train_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map           = "auto",
    trust_remote_code    = True
)
dpo_train_base = prepare_model_for_kbit_training(dpo_train_base)

# Load your SFT LoRA weights on top, then continue training with DPO
dpo_model = PeftModel.from_pretrained(dpo_train_base, SFT_MODEL_DIR, is_trainable=True)
print("✅ SFT model loaded with trainable LoRA for DPO!")

dpo_tokenizer.pad_token    = dpo_tokenizer.eos_token
dpo_tokenizer.padding_side = "left"   # DPO prefers left padding

In [ ]:
dpo_dataset = Dataset.from_list(dpo_pairs)

dpo_config = DPOConfig(
    output_dir                  = DPO_OUTPUT_DIR,
    num_train_epochs            = DPO_EPOCHS,
    per_device_train_batch_size = DPO_BATCH_SIZE,
    gradient_accumulation_steps = DPO_GRAD_ACCUM,
    learning_rate                = DPO_LR,
    beta                         = DPO_BETA,
    bf16                         = True,
    logging_steps                = 5,
    save_steps                   = 50,
    save_total_limit             = 2,
    report_to                    = "none",
    optim                        = "paged_adamw_8bit",
    max_length                   = 512,
    max_prompt_length            = 256,
)

dpo_trainer = DPOTrainer(
    model         = dpo_model,
    args          = dpo_config,
    train_dataset = dpo_dataset,
    processing_class = dpo_tokenizer,
)

print("🏋️  Starting DPO training...")
dpo_trainer.train()

In [ ]:
print("💾 Saving DPO fine-tuned model...")
dpo_trainer.save_model(DPO_OUTPUT_DIR)
dpo_tokenizer.save_pretrained(DPO_OUTPUT_DIR)
print(f"✅ DPO model saved to {DPO_OUTPUT_DIR}")
print("🎉 DPO fine-tuning complete!")
print("\n👉 Don't forget to click 'Save Version' to keep this model permanently!")

In [ ]:
print("Loading DPO fine-tuned model...")
dpo_eval_base  = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
dpo_eval_model = PeftModel.from_pretrained(dpo_eval_base, "/kaggle/working/dpo_finetuned_model")
dpo_eval_model.eval()
dpo_eval_tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/dpo_finetuned_model", trust_remote_code=True)

dpo_results = run_model_on_test_set(dpo_eval_model, dpo_eval_tokenizer, "dpo_finetuned")
with open("/kaggle/working/after_dpo_results.json", "w") as f:
    json.dump(dpo_results, f, indent=2)

del dpo_eval_model, dpo_eval_base
torch.cuda.empty_cache()
print("DPO results saved!")

In [ ]:
dpo_evals  = score_result_set(dpo_results, "AFTER (DPO)")
dpo_scores = [e["gpt_score"] for e in dpo_evals if e.get("gpt_score") is not None]
dpo_avg    = sum(dpo_scores) / len(dpo_scores) if dpo_scores else 0

with open("/kaggle/working/after_dpo_evaluation.json", "w") as f:
    json.dump(dpo_evals, f, indent=2)

In [ ]:
print("\n" + "=" * 65)
print("  FINAL COMPARISON: Original vs SFT vs DPO")
print("=" * 65)
print(f"  Original (base) : {before_avg:.2f} / 10")
print(f"  SFT fine-tuned  : {after_avg:.2f} / 10  ({after_avg-before_avg:+.2f})")
print(f"  DPO fine-tuned  : {dpo_avg:.2f} / 10  ({dpo_avg-before_avg:+.2f} vs original)")
print("=" * 65)

log_event("FINAL_3WAY_COMPARISON", {
        "original_avg": round(before_avg, 2), "sft_avg": round(after_avg, 2),
        "dpo_avg": round(dpo_avg, 2), "bit_precision": BIT_PRECISION,
        "chunk": RESUME_FROM_CHUNK
    }, notes="Full SFT -> DPO pipeline comparison")

# Anytime you want to see everything logged so far, across all sessions:
show_log()
